In [ ]:
%matplotlib widget
%run ./LibStm32Boot.ipynb
%run ./LibLoadSaveSettings.ipynb
%run ./LibLogicAnalyserGlitchTools.ipynb

import numpy as np
import matplotlib.pyplot as plt


FLASH_SIZE = 256      # test
CHUNK = 64                   # SAM3U RX buffer holds only 128-200 bytes
BAUD = 115200
BYTE_GAP = 0.0001              # back-to-back bytes get NACKed by the bootloader
RETRIES = 3
debug=False

out_path = "flash_dump.bin"
size = FLASH_SIZE

try:
    scope.dis()
except:
    pass
try:
    fig1.dis()
except:
    pass
try:
    fig2.dis()
except:
    pass

def init_logic_analyser(scope, glitch=True, capture_group='CW 20-pin'):
    scope.errors.clear()
    scope.adc.segments = 1
    scope.gain.db = 25

    scope.io.hs2 = 'glitch'              # Glitch on HS2, for example to feed an external device
    scope.io.glitch_trig_mcx = 'glitch' # Glitch pulse on the Trigger/Glitch Out MCX connector
    scope.io.tio1 = "serial_tx"
    scope.io.tio2 = "serial_rx"
    scope.io.tio3 = "gpio_high"

    scope.trigger.module = "basic"
    scope.trigger.triggers = "tio1"
    scope.glitch.trigger_src = "ext_single"   # glitch also fires on that trigger

    scope.adc.basic_mode = "falling_edge"   # catch the actual start bit
    scope.adc.presamples = 0
    scope.adc.samples = 10
    scope.adc.lo_gain_errors_disabled = True
    
    # --- Logic analyzer: record trigger + glitch signals ---
    scope.LA.enabled = True
    scope.LA.capture_group = capture_group
    scope.LA.capture_depth = 40000
    scope.LA.trigger_source = 'capture'     
    scope.LA.oversampling_factor = 20
    scope.LA.downsample = 1   # try 100 to 1000 until you see the whole exchange

    if glitch:
        scope.glitch.enabled = True
        scope.io.glitch_hp = True
        scope.io.glitch_lp = True
    else:
        scope.glitch.enabled = False
        scope.io.glitch_hp = False
        scope.io.glitch_lp = False

    scope.glitch.clk_src = 'pll'
    #scope.glitch.output = "enable_only"  # one large glitch repeat times
    scope.glitch.output = "glitch_only" # multiple short (width) glitches
    


scope = cw.scope()
scope.default_setup(verbose=False)
target = cw.target(scope, cw.targets.SimpleSerial)
set_debug_pin(True)

scope.clock.clkgen_freq = 6e6
ser = target.ser
ser.setBaud(BAUD)
ser.setParity("even")

init_logic_analyser(scope)


enter_bootloader()

# Get ID as a sanity check (expect 0462 for the L45x family)
get_product_id()

def trigger_operation():
    data = []
    try:
        data = read_mem(addr=FLASH_BASE, length=256)
    except Exception as e:
        print(f"exception {e}")
    return data

def pre_arm_action(scope):
    pass

def post_arm_trigger_action(scope):
    trigger_operation()

def post_capture_action(scope):
    pass


def capture_with_la(scope, capture_group, addr=FLASH_BASE, length=256, ext_offset=10, offset=0, width=3000, repeat=1, trigger='trigger_glitch'):
    scope.errors.clear()

    scope.glitch.offset = offset
    scope.glitch.ext_offset = ext_offset
    scope.glitch.width = width
    scope.glitch.repeat = repeat

    print("scope.glitch.width ", scope.glitch.width)

    scope.LA.capture_group = capture_group
    ser.flush()
    time.sleep(0.01)
    scope.LA.enabled = True
    scope.LA.arm()   
    scope.arm()
    print("scope.glitch.mmcm_locked ", scope.glitch.mmcm_locked)
    time.sleep(0.003)
    data = trigger_operation()
    
    res = read_capture_data(scope)

    scope.LA.enabled = False
    return res, data

import numpy as np

def get_la_sample_period(scope):
    """Sample period in seconds. Uses the LA sampling clock if available."""
    la = scope.LA
    f = getattr(la, "sampling_clock_frequency", None)   # Hz
    if not f:                                            # fallback: clkgen * oversampling
        f = scope.clock.clkgen_freq * la.oversampling_factor
    return 1.0 / f

import numpy as np

def on_times(sig, sample_period_s, include_partial=False):
    """
    On (high) times of all pulses in a 0/1 array.

    include_partial: also count pulses cut off at the start/end of the capture
                     (their true length is unknown, so the result is a lower bound)
    """
    sig = np.asarray(sig).astype(np.int8)
    d = np.diff(sig)
    rising  = np.where(d == 1)[0] + 1    # first high sample
    falling = np.where(d == -1)[0] + 1   # first low sample after the pulse

    starts_high = sig[0] == 1
    ends_high   = sig[-1] == 1

    if starts_high:                      # pulse already running at sample 0
        if include_partial:
            rising = np.r_[0, rising]
        else:
            falling = falling[1:]        # drop the falling edge of the partial pulse
    if ends_high:                        # pulse still running at the last sample
        if include_partial:
            falling = np.r_[falling, len(sig)]
        else:
            rising = rising[:-1]         # drop the rising edge of the partial pulse

    n = min(len(rising), len(falling))
    rising, falling = rising[:n], falling[:n]

    width_samples = falling - rising
    return {
        "rising_idx": rising,
        "falling_idx": falling,
        "width_samples": width_samples,
        "width_ns": width_samples * sample_period_s * 1e9,
        "start_ns": rising * sample_period_s * 1e9,
    }

def measure_on_time(cap, channel, include_partial=False, verbose=True):
    """
    Measure the on time of one signal from the result of read_capture_data().

    cap     : dict returned by read_capture_data(scope)
    channel : signal name (e.g. "HS2", case-insensitive) or index into cap["sigs"]
    """
    names = cap["names"]
    if isinstance(channel, str):
        lower = [str(n).lower() for n in names]
        if channel.lower() not in lower:
            raise ValueError(f"Signal '{channel}' not in capture group: {names}")
        idx = lower.index(channel.lower())
    else:
        idx = int(channel)

    t = cap["time_axis_s"]
    if len(t) < 2:
        raise ValueError("Capture too short")
    sample_period_s = t[1] - t[0]        # = downsample / sampling_clock_frequency

    res = on_times(cap["sigs"][idx], sample_period_s, include_partial)
    res["name"] = names[idx]
    res["sample_period_ns"] = sample_period_s * 1e9

    if verbose:
        w = res["width_ns"]
        print(f"{names[idx]}: sample period {res['sample_period_ns']:.3f} ns")
        if len(w) == 0:
            print("  no complete high pulse found")
        else:
            print(f"  pulses: {len(w)}  mean: {w.mean():.2f} ns  min: {w.min():.2f} ns  max: {w.max():.2f} ns")
            for s, wd in zip(res["start_ns"], w):
                print(f"    start {s:.2f} ns  on-time {wd:.2f} ns")
    return res

# Usage
# scope.LA.arm()
# ... trigger the event ...
# cap = read_capture_data(scope)
# res = measure_on_time(cap, "HS2")     # by name
# res = measure_on_time(cap, 0)         # by index


off=0
addr = FLASH_BASE + off
length = min(CHUNK, size - off)

# 1350 - 1500
ext_offset = 1100
width = 2800
offset = 0
repeat = 1

res, data = capture_with_la(scope, 'CW 20-pin', addr, length, ext_offset=ext_offset, width=width, offset=offset, repeat=repeat)
save_scope_settings(scope, "first_capture.json")

f_la = scope.LA.sampling_clock_frequency / scope.LA.downsample
fig1 = plot_la_interactive(res, title=caption(calc_glitch_times(scope)))
mesured_pulse = measure_on_time(res, "HS2")

res, data = capture_with_la(scope, 'glitch', ext_offset=ext_offset, width=width, offset=offset, repeat=repeat)
save_scope_settings(scope, "second_capture.json")

fig2 = plot_la_interactive(res, title=caption(calc_glitch_times(scope)))


scope.dis()
scope = None


scope.glitch.width  2800
scope.glitch.mmcm_locked  True
exception Read Memory NACK
